# Amparo -- Baseline y fine-tuning LoRA del agente legal

Este notebook:
1. Carga `dataset_legal.jsonl` (1320 ejemplos en 24 categorias) y separa una validacion estratificada por categoria que **nunca** se usa para entrenar.
2. Corre el modelo base sin modificar sobre esa validacion (**baseline**).
3. Entrena un adaptador **LoRA** sobre el resto del dataset.
4. Vuelve a correr el modelo ya afinado sobre la misma validacion.
5. Compara baseline vs. afinado -- esa comparacion es la evidencia real de que el fine-tuning mejoro algo, no solo una suposicion.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU` (una T4 gratis alcanza usando 4-bit/QLoRA).

**Recomendacion:** usa `Entorno de ejecucion > Ejecutar todas` en vez de correr celdas sueltas, para evitar errores de variables no definidas por saltarte una celda.

**Nota sobre las librerias:** `transformers`, `peft` y `trl` cambian su API con cierta frecuencia. Este notebook esta escrito contra las versiones estables mas recientes conocidas al momento de escribirlo; si algun cambio de firma da error, revisa el changelog de la libreria correspondiente -- la logica de cada celda (que hace y por que) sigue siendo valida aunque cambie algun nombre de parametro.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets wandb

## Configuracion

Cambia `MODEL_ID` para comparar Qwen2.5 7B vs. Llama 3.1 8B (o el que haya ganado en el comparador local `tools/model_comparator`).

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # alternativa: "meta-llama/Llama-3.1-8B-Instruct"

VAL_FRACTION = 0.15   # % por categoria reservado para validacion (nunca se entrena con esto)
RANDOM_SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8   # batch efectivo = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
MAX_SEQ_LENGTH = 1024

MAX_NEW_TOKENS_EVAL = 300
OUTPUT_DIR = "/content/amparo-lora"

## Monta tu Google Drive y ubica el dataset

Sube `dataset_legal.jsonl` una sola vez a `MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl` (ajusta `DATASET_PATH` si usaste otra ruta). Con Drive no tienes que volver a subir el archivo cada vez que el entorno de Colab se reinicia.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl'  # ajusta si usaste otra ruta
print(f'Usando dataset: {DATASET_PATH}')

## Cargar y separar train / validacion (estratificado por categoria)

In [ ]:
import json
import random
from collections import defaultdict

random.seed(RANDOM_SEED)

records = [json.loads(line) for line in open(DATASET_PATH, encoding='utf-8')]

by_category = defaultdict(list)
for r in records:
    by_category[r['category']].append(r)

train_records, val_records = [], []
for category, items in by_category.items():
    items = items[:]
    random.shuffle(items)
    n_val = max(1, round(len(items) * VAL_FRACTION))
    val_records.extend(items[:n_val])
    train_records.extend(items[n_val:])

random.shuffle(train_records)
random.shuffle(val_records)

print(f'Total: {len(records)} | Train: {len(train_records)} | Validacion: {len(val_records)}')
for category in sorted(by_category):
    n_val = sum(1 for r in val_records if r['category'] == category)
    n_train = sum(1 for r in train_records if r['category'] == category)
    print(f'  {category:45s} train={n_train:3d}  val={n_val:3d}')

## Cargar el modelo base en 4-bit (QLoRA)

Se carga cuantizado en 4-bit para que quepa comodamente en una GPU gratuita de Colab (T4, ~16GB).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

## Metrica de similitud

Misma heuristica lexica (difflib) que usa el comparador local en `tools/model_comparator/metrics.py`, para que los resultados sean comparables entre ambas herramientas. No es una metrica juridica rigurosa de correccion legal, solo sirve para ordenar/comparar respuestas de forma consistente.

In [ ]:
from difflib import SequenceMatcher

SYSTEM_PROMPT = records[0]['messages'][0]['content']


def similarity_pct(expected: str, actual: str) -> float:
    if not expected or not actual:
        return 0.0
    ratio = SequenceMatcher(None, expected.strip().lower(), actual.strip().lower()).ratio()
    return round(ratio * 100, 1)


@torch.no_grad()
def generate_response(model, query: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_EVAL,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def evaluate(model, val_set, label: str):
    results = []
    for i, item in enumerate(val_set):
        query = item['messages'][1]['content']
        expected = item['messages'][2]['content']
        actual = generate_response(model, query)
        sim = similarity_pct(expected, actual)
        results.append({**item, 'generated': actual, 'similarity': sim})
        print(f"[{label}] {i + 1}/{len(val_set)}  sim={sim:5.1f}  {item['category']}")
    avg = sum(r['similarity'] for r in results) / len(results)
    print(f'\n[{label}] Similitud promedio: {avg:.1f}')
    return results

## Paso 1 -- Baseline (modelo base, sin fine-tuning)

Esto puede tardar varios minutos segun el tamano de la validacion.

In [ ]:
baseline_results = evaluate(model, val_records, label='baseline')

with open('/content/baseline_results.jsonl', 'w', encoding='utf-8') as f:
    for r in baseline_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

## Paso 2 -- Fine-tuning con LoRA

### Antes de entrenar: conecta Weights & Biases

M1 pide dejar registrada la curva de perdida del entrenamiento como evidencia visual de que el modelo esta aprendiendo. Crea una cuenta gratuita en wandb.ai si no tienes una, y ten a mano tu API key (la consigues en https://wandb.ai/authorize).

In [ ]:
import wandb

wandb.login()  # pega tu API key cuando te la pida (solo la primera vez)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset


def format_for_training(record):
    return {'text': tokenizer.apply_chat_template(record['messages'], tokenize=False)}


train_dataset = Dataset.from_list(train_records).map(format_for_training)

In [ ]:
wandb.init(
    project='amparo-legal-finetune',
    name=f"lora-{MODEL_ID.split('/')[-1]}",
    config={
        'model_id': MODEL_ID,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
    },
)

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    report_to='wandb',
    run_name=f"lora-{MODEL_ID.split('/')[-1]}",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

print('Dashboard de W&B (guarda este link para tu informe de M1):', wandb.run.url)
wandb.finish()

## Guardar el adaptador LoRA en Drive

Se guarda directo en tu Drive (no solo son unos MB, son los pesos LoRA) para que no se pierda si el entorno de Colab se desconecta a mitad del entrenamiento o despues.

In [ ]:
import shutil

model.save_pretrained(f'{OUTPUT_DIR}/adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/adapter')

DRIVE_ADAPTER_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter'
shutil.copytree(f'{OUTPUT_DIR}/adapter', DRIVE_ADAPTER_DIR, dirs_exist_ok=True)
print(f'Adaptador guardado en {DRIVE_ADAPTER_DIR}')

## Paso 3 -- Evaluar el modelo ya afinado (misma validacion)

In [ ]:
finetuned_results = evaluate(model, val_records, label='fine-tuned')

with open('/content/finetuned_results.jsonl', 'w', encoding='utf-8') as f:
    for r in finetuned_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

## Paso 4 -- Comparacion baseline vs. afinado

Esta tabla es la evidencia real (no una suposicion) de si el fine-tuning mejoro las respuestas, por categoria y en promedio general.

In [ ]:
import pandas as pd

df_base = pd.DataFrame(baseline_results)[['id', 'category', 'similarity']].rename(columns={'similarity': 'baseline'})
df_ft = pd.DataFrame(finetuned_results)[['id', 'similarity']].rename(columns={'similarity': 'fine_tuned'})
comparison = df_base.merge(df_ft, on='id')
comparison['mejora'] = comparison['fine_tuned'] - comparison['baseline']

summary = comparison.groupby('category')[['baseline', 'fine_tuned', 'mejora']].mean().round(1)
print(summary)
print(
    f"\nPromedio general -> baseline: {comparison['baseline'].mean():.1f}  "
    f"fine-tuned: {comparison['fine_tuned'].mean():.1f}  "
    f"mejora: {comparison['mejora'].mean():+.1f}"
)

comparison.to_csv('/content/comparacion_baseline_vs_finetuned.csv', index=False)